In [13]:
import json
import os

import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from openai import OpenAI
from dotenv import load_dotenv

from utils.data import edinet_to_industry_map, all_securities, jp_500
from utils.datetime import date_string_to_quarter
from utils.edinet_api import get_doc_name
from utils.ELO import EloRatingSystem, get_num_games, generate_random_pdf_pair
from utils.signal import get_winner, get_winner_grok_mini, get_stock_code_from_path

from utils.datapath import (
    documents_path,
    edinet_codes_path,
    docs_metadata_path,
    industry_elo_signals_path
)

In [14]:
load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

grok_client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

In [15]:
df = pd.read_excel(edinet_codes_path)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [16]:
top_security_codes = list(set([code for quarter, codes in jp_500.items() for code in codes]))
top_securities = [security for security in all_securities if security["code"] in top_security_codes]
top_securities_edinet = [security["edinet_code"] for security in top_securities]
len(top_securities)

740

In [17]:
filtered_doc_metadata = [doc for doc in docs_metadata if doc["edinetCode"] in top_securities_edinet]
len(filtered_doc_metadata)

20175

In [18]:
industry_quarterly_docs = defaultdict(lambda: defaultdict(list))

for doc in filtered_doc_metadata:
    industry = edinet_to_industry_map.get(doc['edinetCode'], "na")
    period_end_quater = date_string_to_quarter(doc["periodEnd"])

    industry_path = os.path.join(documents_path, industry)
    quarter_path = os.path.join(industry_path, period_end_quater)

    save_name = get_doc_name(doc)
    output_path = os.path.join(quarter_path, save_name)

    industry_quarterly_docs[industry][period_end_quater].append(output_path)

In [19]:
selected_industries = [
    # "Real Estate",
    # "Precision Instruments",
    # "Electric Power & Gas",
    # "Other Financing Business",
    # "Glass & Ceramics Products",
    # "Nonferrous Metals",
    # "Securities & Commodity Futures",
    "Iron & Steel",
    "Metal Products",
]
# selected_industries = industry_quarterly_docs.keys()

In [8]:
industry_quarterly_signals = {}

In [20]:
industry = "Iron & Steel"
quarterly_docs = industry_quarterly_docs[industry]

client = grok_client
win_fn = get_winner_grok_mini

iron_n_steel_games = []
iron_n_steel_elo_system = EloRatingSystem(ratings={})
quarterly_signals = {}

# Calculate total iterations (quarters * games per quarter)
total_games = sum(get_num_games(len(docs)) for docs in quarterly_docs.values())
progress_bar = tqdm(total=total_games, desc="Playing games")

for quarter, docs in quarterly_docs.items():
    for pdf1_path, pdf2_path in generate_random_pdf_pair(
        docs, get_num_games(len(docs))
    ):
        try:
            winner = win_fn(client, pdf1_path, pdf2_path)
        except:
            print(f"Grok failed for {pdf1_path} vs {pdf2_path}")

        companyA_code, companyB_code = [
            get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
        ]
        iron_n_steel_elo_system.update_ratings(companyA_code, companyB_code, winner)
        iron_n_steel_games.append((quarter, companyA_code, companyB_code, winner))
        progress_bar.update(1)

    quarterly_signals[quarter] = iron_n_steel_elo_system.get_all_ratings()

progress_bar.close()
industry_quarterly_signals[industry] = quarterly_signals

Playing games:   9%|▉         | 70/772 [10:47<1:47:46,  9.21s/it]

Grok failed for ../../documents/Iron & Steel/2015-Q4/E01239_大同特殊鋼株式会社_140_S1006X76.pdf vs ../../documents/Iron & Steel/2015-Q4/E01264_ＪＦＥホールディングス株式会社_140_S1006SYU.pdf


Playing games:  11%|█▏        | 87/772 [13:27<1:48:04,  9.47s/it]

Grok failed for ../../documents/Iron & Steel/2016-Q2/E01259_大和工業株式会社_140_S1008DNY.pdf vs ../../documents/Iron & Steel/2016-Q2/E01225_日本製鉄株式会社_140_S1008DOX.pdf


Playing games:  12%|█▏        | 91/772 [14:04<1:46:19,  9.37s/it]

...

Playing games:  88%|████████▊ | 678/772 [1:43:22<14:02,  8.96s/it]

Grok failed for ../../documents/Iron & Steel/2023-Q2/E01231_株式会社　神戸製鋼所_140_S100RK4J.pdf vs ../../documents/Iron & Steel/2023-Q2/E01264_ＪＦＥホールディングス株式会社_140_S100RI8R.pdf


Playing games:  90%|█████████ | 695/772 [1:45:44<11:03,  8.61s/it]

Grok failed for ../../documents/Iron & Steel/2023-Q2/E01259_大和工業株式会社_140_S100RN2W.pdf vs ../../documents/Iron & Steel/2023-Q2/E01243_山陽特殊製鋼株式会社_140_S100RLKD.pdf


Playing games: 100%|██████████| 772/772 [1:57:54<00:00,  9.16s/it]


In [21]:
industry = "Metal Products"
quarterly_docs = industry_quarterly_docs[industry]

client = grok_client
win_fn = get_winner_grok_mini

metal_products_games = []
metal_products_elo_system = EloRatingSystem(ratings={})
quarterly_signals = {}

# Calculate total iterations (quarters * games per quarter)
total_games = sum(get_num_games(len(docs)) for docs in quarterly_docs.values())
progress_bar = tqdm(total=total_games, desc="Playing games")

for quarter, docs in quarterly_docs.items():
    for pdf1_path, pdf2_path in generate_random_pdf_pair(
        docs, get_num_games(len(docs))
    ):
        try:
            winner = win_fn(client, pdf1_path, pdf2_path)
        except:
            print(f"Grok failed for {pdf1_path} vs {pdf2_path}")

        companyA_code, companyB_code = [
            get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
        ]
        metal_products_elo_system.update_ratings(companyA_code, companyB_code, winner)
        metal_products_games.append((quarter, companyA_code, companyB_code, winner))
        progress_bar.update(1)

    quarterly_signals[quarter] = metal_products_elo_system.get_all_ratings()

progress_bar.close()
industry_quarterly_signals[industry] = quarterly_signals

Playing games:   6%|▋         | 33/528 [04:30<1:12:42,  8.81s/it]

Grok failed for ../../documents/Metal Products/2015-Q3/E01353_東洋製罐グループホールディングス株式会社_140_S1006CG4.pdf vs ../../documents/Metal Products/2015-Q3/E02103_株式会社ＳＵＭＣＯ_140_S10069LC.pdf


Playing games:  17%|█▋        | 90/528 [12:42<1:07:18,  9.22s/it]

Grok failed for ../../documents/Metal Products/2016-Q3/E01353_東洋製罐グループホールディングス株式会社_140_S1009317.pdf vs ../../documents/Metal Products/2016-Q3/E01317_株式会社ＬＩＸＩＬ_140_S100934L.pdf


Playing games:  22%|██▏       | 116/528 [16:39<1:01:20,  8.93s/it]

...

Playing games:  37%|███▋      | 193/528 [28:15<1:00:07, 10.77s/it]

Grok failed for ../../documents/Metal Products/2018-Q3/E01353_東洋製罐グループホールディングス株式会社_140_S100EJRK.pdf vs ../../documents/Metal Products/2018-Q3/E01317_株式会社ＬＩＸＩＬ_140_S100EFT9.pdf


Playing games:  61%|██████    | 323/528 [48:01<32:51,  9.62s/it]  

Grok failed for ../../documents/Metal Products/2020-Q3/E01317_株式会社ＬＩＸＩＬグループ_140_S100JZAL.pdf vs ../../documents/Metal Products/2020-Q3/E01382_東プレ株式会社_140_S100K5RH.pdf


Playing games: 100%|██████████| 528/528 [1:17:48<00:00,  8.84s/it]


In [22]:
for industry, quarterly_docs in tqdm(industry_quarterly_docs.items()):
    # client = None if industry not in selected_industries else grok_client
    # win_fn = get_winner if industry not in selected_industries else get_winner_grok_mini

    if industry in selected_industries:
        continue

    client = None
    win_fn = get_winner
    
    elo_system = EloRatingSystem(ratings={})
    quarterly_signals = {}

    for quarter, docs in quarterly_docs.items():
        for pdf1_path, pdf2_path in generate_random_pdf_pair(
            docs, get_num_games(len(docs))
        ):
            winner = win_fn(client, pdf1_path, pdf2_path)
            companyA_code, companyB_code = [
                get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
            ]
            elo_system.update_ratings(companyA_code, companyB_code, winner)

        quarterly_signals[quarter] = elo_system.get_all_ratings()
    
    industry_quarterly_signals[industry] = quarterly_signals


100%|██████████| 33/33 [00:01<00:00, 26.07it/s]


In [23]:
# industry_quarterly_signals.get("Metal Products")
industry_quarterly_signals.get("Iron & Steel")
# industry_quarterly_signals.get("Marine Transportation")


{'2015-Q2': {'5481': 1438.3978832375349,
  '5471': 1591.3722433028079,
  '5411': 1465.5702371826114,
  '5444': 1527.0156983169009,
  '5401': 1391.9289941325505,
  '5463': 1560.4574220957481,
  '5423': 1505.7918588020777,
  '5406': 1519.4656629297685},
 '2015-Q3': {'5481': 1481.2483708021928,
  '5471': 1543.5200144300586,
  '5411': 1415.884881542789,
  '5444': 1505.6358044539672,
  '5401': 1461.3257642165013,
  '5463': 1601.7674756853294,
  '5423': 1440.008305911225,
  '5406': 1550.6093829579368},
 '2015-Q4': {'5481': 1390.0240433385625,
  '5471': 1499.8731989925752,
  '5411': 1487.0580961749686,
  '5444': 1539.922649828896,
  '5401': 1485.0214480304448,
  '5463': 1595.0646897120419,
  '5423': 1374.7302999783626,
  '5406': 1628.3055739441484},
 '2016-Q2': {'5481': 1493.346109227706,
  '5471': 1387.740646369721,
  '5411': 1506.3463930096757,
  '5444': 1433.619850822273,
  '5401': 1494.1962603982686,
  '5463': 1574.873648489472,
  '5423': 1463.4878055609736,
  '5406': 1646.3892861219101},

In [ ]:
# request_count

# Cost Analysis
model_input_prices = {
    "grok-3-mini": 0.3,
    "grok-3": 3,
    "deepseek-chat": 0.27,
    "deepseek-chat-discount": 0.135,
    "deepseek-reasoner": 0.55,
    "deepseek-reasoner-discount": 0.135,
}

num_requests = 0
model = "grok-3-mini"
num_input_token_per_request = 75000
price_per_million_input_token = model_input_prices[model] # US dollars 
price_per_request = (num_input_token_per_request / 1e6) * price_per_million_input_token
total_price = price_per_request * num_requests
print(f"Total cost per run: ${total_price:.2f}")

Total cost per run: $29.25


In [58]:
num_requests

1300

In [ ]:
# Industry wise elo - DONE
# Industry wise portfolio backtest - DONE

# But that's not the important point 
# Get to the api calls!!!!!!!!!

In [24]:
# Industry wise elo

with open(industry_elo_signals_path, 'w', encoding='utf-8') as f:
    json.dump(industry_quarterly_signals, f)



In [27]:
# Deepseek >50% off from 12:30 to 8:30 am HKT
notebook_to_root = os.path.join("..", "..")
signals_path = os.path.join(notebook_to_root, "signals")
iron_n_steel_games_path = os.path.join(signals_path, "iron_n_steel_games.json")

with open(iron_n_steel_games_path, 'w', encoding='utf-8') as f:
    json.dump(iron_n_steel_games_path, f)

In [28]:
metal_products_games
notebook_to_root = os.path.join("..", "..")
signals_path = os.path.join(notebook_to_root, "signals")
metal_products_games_path = os.path.join(signals_path, "metal_products_games.json")

with open(metal_products_games_path, 'w', encoding='utf-8') as f:
    json.dump(metal_products_games_path, f)